In [1]:
import ast
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
from shutil import copyfile
import re

In [2]:
table_dir = '../data/data_in_sep_pkl/'
assert os.path.exists(table_dir)

In [3]:
paths = []
for f in os.listdir(table_dir):
    paths.append(os.path.join(table_dir, f))
paths = sorted(paths)
tot = len(paths)
print(tot)

2009


In [4]:
def get_json_dict(i):
    d = pickle.load(open(os.path.join(paths[i], 'init.pkl'), 'rb'))
    return d

In [5]:
def extra_unit_check(unit):
    
    if len(unit)==0 or not unit[0].isalpha() or len(unit)>10 : return False
    
    out_words = ['nominal', 'ordinal', 'experimental']
    if unit.lower() in out_words:
        return False
    
    un_list = [int(s) for s in re.findall(r'\d+', unit)]
    if len(un_list)>0:
        for uni in un_list:
            #print(uni)
            if uni>10:
                return False
    return True

In [6]:
prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 70, 1306]
prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'BulkModulus', 'ActivationEnergy']
unit_dict = {prop: set() for prop in prop_names}
assert len(prop_ids) == len(prop_names)

In [7]:
def set_units(i):
    #global qgrid_widget_0
    #display(qgrid_widget_0)
    d = get_json_dict(i)
    #print(d)
    r, c = d['num_rows'], d['num_cols']
    table_name = d['pii'] + '__' + str(d['t_idx'])
#     print(f'Table Name : {table_name}')
#     print(f'Comp table: {d["comp_table"]}')
#     print(f'Prop table: {d["prop_table"]}')
#     print(f'Prop names: {d["prop_names"]}')
#     print(f'Prop orient: {d["prop_orient"]}')
#     print(f'Prop index: {d["prop_row_col_indexes"]}')
    # print(f'Props: {d["props"]}')
    #prop_names = ["d", "tg", "ri", "abbe"]
    #prop_ids = [510, 1140, 2010, 2051, 540, 50, 180, 60, 160, 1014, 1015, 3012, 3174, 1116, 1119, 1020, 1011, 1118, 70, 1306]
    #prop_names = ['Density', 'GlassTransitionTg', 'RefractiveIndex', 'AbbeValue', 'YoungsModulus', 'ShearModulus', 'VickersHardness', 'PoissonRatio', 'FractureToughness', 'CrystallizationTemp', 'MeltingTemp', 'ElectricConduct', 'DielectricConst', 'TSofP', 'TAnnealingP', 'ExpansionCoeff', 'LiquidusTemperature', 'SofP', 'BulkModulus', 'ActivationEnergy']
    prop_names_in_tables = []
    #unit_dict = {prop: set() for prop in prop_names}
    
    if "prop_row_col_indexes" not in d:
        print(f'prop_row_col_indexes missing {i}')
        return
    
    for inde, elem in enumerate(d["prop_row_col_indexes"]):
        if len(elem) != 0:
            for je in elem:
                prop_names_in_tables.append(prop_names[inde])
    #print(f'list of prop_names = {prop_names_in_tables}')
    #co = 0
    for ind, single_prop_table in enumerate(d["props"]):
        if len(d["prop_row_col_indexes"][ind]) != 0:
            #display(pd.DataFrame(single_prop_table))
            spt = single_prop_table
#             assert len(d["prop_orient"])==len(prop_names_in_tables), 'Dimensions not matching'
            prop_name = prop_names[ind]
#             print(prop_name)
#             print(list(d["prop_names"])[co])

            
            if d["prop_orient"]=='col':
                #print('ENETERED')
                for prop_col in d["prop_row_col_indexes"][ind]:    
                    #prop_col = d["prop_row_col_indexes"][ind][0]
                    spt_t = np.array(d['act_table'])
                    spt_c = spt_t[:,prop_col].tolist()
#                     print(spt_c)
                    ind_c = 0
                    unit = ''
                    pr_spt_t = np.array(d['props'][ind])
                    pr_spt_c = pr_spt_t[:,prop_col].tolist()
                    spt_pr = ''
                    for indd, ele in enumerate(pr_spt_c):
                        if len(ele)!=0:
                            ind_c = indd
#                             print(f'ind_c = {ind_c}')
                            break
#                     for indd in range(ind_c-1, -1, -1):
                    for indd in range(0, ind_c):
#                         print('ENETERED')
                        #print(f'spt_c[indd] = {spt_c[indd]}')
                        ex_strr = spt_c[indd]
    
                        if re.search('[\(|\[]\s?\+\-\s?[0-9]+\s?\.?[0-9]*\s?[a-zA-Z]+\s?[\)|\]]', ex_strr) is not None:
                            ex_strr = re.sub('\s?\+\-\s?[0-9]+\s?\.?[0-9]*\s?', '', ex_strr)
            
                        ex_str = re.search('[\(|\[]\s?[a-zA-Z|\-|\s|0-9|\\|\/\|\°]+[\)|\]]', ex_strr)

                        #print(f'ex_str == {ex_str}')
                        if ex_str is not None:
                            unit = ex_str.group()[1:-1].strip()
                            unit = re.sub('\s+', ' ', unit)
                            #unit = re.sub('\+\-\s?[0-9]+\s?', '', unit)
                        
                            ##Only property specific constraint in the program
                            if prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP'] and unit[len(unit)-1].upper() == 'C':
                                unit = 'degC'
                            elif  prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP'] and unit[len(unit)-1].upper() == 'K':
                                unit = 'K'
                                
                            if prop_name in ['ExpansionCoeff'] and 'C-1' in unit.upper():
                                unit = 'degC-1'
                            elif  prop_name in ['ExpansionCoeff'] and 'K-1' in unit.upper():
                                unit = 'K-1'
                        
                            if extra_unit_check(unit):
                                break
                            else:
                                unit = ''
                                
                            
                    if unit=='':
                        if prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP']: 
                            unit = 'degC'
                        if prop_name in ['ExpansionCoeff']:
                            unit = 'degC-1'
                        
                    if prop_name in prop_name in ['RefractiveIndex', 'AbbeValue', 'PoissonRatio','DielectricConst']: 
                        unit = None
                    
#                     print(f'unit == {unit}')
#                     print(prop_name)
                    
                    for indd in range(ind_c, r):
                        em_list=[]
                        for elem in pr_spt_c[indd]:
#                             print(f'pr_spt_c[indd] = {pr_spt_c[indd]}')
#                             print(f'elem = {elem}')
                            elem = list(elem)
                            
                            if unit=='' and prop_name=='Density':
#                                 print(d['anno_type'])
#                                 print(elem[1])
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val<23: unit = 'gm/cm3'
                                else: unit = 'kg/m3'
                            elif unit=='' and prop_name in ['YoungsModulus', 'ShearModulus', 'BulkModulus']:
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val<=300: unit = 'GPa'
                                elif c_val>=1000: unit = 'MPa'
                            elif unit not in ['kJmol-1', 'kJ mol-1','kJ/mol', 'kJ / mol', 'kJ/molK', 'eV/at', 'eV', 'kcal/mol', 'kcalmol-1', 'kcal mol-1', 'Jmol-1', 'J mol-1','J/mol', 'J / mol'] and prop_name == 'ActivationEnergy':
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val>=20: unit = 'kJ/mol'
                            
                            elem[2] = unit
                            unit_dict[prop_name].add(unit)
                            #print(elem)
                            em_list.append(tuple(elem))
                            
                        if len(em_list)>1:
#                             print(table_name)
#                             print(em_list)
#                             print(d['anno_type'])
                            em_list = [em_list[0]]
                        
                        d['props'][ind][indd][prop_col] = em_list
                        #print(pr_spt_c[indd])
                        
                        
            elif d["prop_orient"]=='row':
                #print('ENETERED')
                for prop_row in d["prop_row_col_indexes"][ind]:    
                    #prop_col = d["prop_row_col_indexes"][ind][0]
                    spt_t = np.array(d['act_table'])
                    spt_c = spt_t[prop_row, :].tolist()
#                     print(spt_c)
                    ind_c = 0
                    unit = ''
                    pr_spt_t = np.array(d['props'][ind])
                    pr_spt_c = pr_spt_t[prop_row, :].tolist()
                    spt_pr = ''
                    for indd, ele in enumerate(pr_spt_c):
                        if len(ele)!=0:
                            ind_c = indd
#                             print(f'ind_c = {ind_c}')
                            break
#                     for indd in range(ind_c-1, -1, -1):
                    
                    for indd in range(0, ind_c):
#                         print('ENETERED')
#                         print(f'spt_c[indd] = {spt_c[indd]}')
                        ex_strr = spt_c[indd]
    
                        if re.search('[\(|\[]\s?\+\-\s?[0-9]+\s?\.?[0-9]*\s?[a-zA-Z]+\s?[\)|\]]', ex_strr) is not None:
                            ex_strr = re.sub('\s?\+\-\s?[0-9]+\s?\.?[0-9]*\s?', '', ex_strr)
            
                        ex_str = re.search('[\(|\[]\s?[a-zA-Z|\-|\s|0-9|\\|\/\|\°]+[\)|\]]', ex_strr)
    
                                    
                        if ex_str is not None:
                            unit = ex_str.group()[1:-1].strip()
                            unit = re.sub('\s+', ' ', unit)
                            #unit = re.sub('\+\-\s?[0-9]+\s?', '', unit)
                        
                            ##Only property specific constraint in the program
                            if prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP'] and unit[len(unit)-1].upper() == 'C':
                                unit = 'degC'
                            elif  prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP'] and unit[len(unit)-1].upper() == 'K':
                                unit = 'K'
                                
                            if prop_name in ['ExpansionCoeff'] and 'C-1' in unit.upper():
                                unit = 'degC-1'
                            elif  prop_name in ['ExpansionCoeff'] and 'K-1' in unit.upper():
                                unit = 'K-1'
                        
                            if extra_unit_check(unit):
                                break
                            else:
                                unit = ''
                    
                    
                    if unit=='':
                        if prop_name in ['GlassTransitionTg', 'CrystallizationTemp', 'MeltingTemp', 'TSofP', 'TAnnealingP', 'LiquidusTemperature','SofP']: 
                            unit = 'degC'
                        if prop_name in ['ExpansionCoeff']:
                            unit = 'degC-1'
                    
                    if prop_name in ['RefractiveIndex', 'AbbeValue', 'PoissonRatio','DielectricConst']:
                        unit = None
                    
                    
                    for indd in range(ind_c, c):
                        em_list = []
                        for elem in pr_spt_c[indd]:
                            elem = list(elem)
                            
#                             print(f'unit = {unit}')
#                             print(f'prop_name = {prop_name}')
                            if unit=='' and prop_name=='Density':
#                                 print('AAAAAAAAAAAAAAAAAA')
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val<23: unit = 'gm/cm3'
                                else: unit = 'kg/m3'
                            elif unit=='' and prop_name in ['YoungsModulus', 'ShearModulus', 'BulkModulus']:
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val<=300: unit = 'GPa'
                                elif c_val>=1000: unit = 'MPa'
                            elif unit not in ['kJmol-1', 'kJ mol-1','kJ/mol', 'kJ / mol', 'kJ/molK', 'eV/at', 'eV', 'kcal/mol', 'kcalmol-1', 'kcal mol-1', 'Jmol-1', 'J mol-1','J/mol', 'J / mol'] and prop_name == 'ActivationEnergy':
                                c_val = elem[1]
#                                 print(f'c_val = {c_val}')
                                if c_val>=20: unit = 'kJ/mol'
                
#                             print(table_name)
#                             print(elem)
                            elem[2] = unit
                            unit_dict[prop_name].add(unit)
                            em_list.append(tuple(elem))
#                             print(em_list)
                                
                        if len(em_list)>1:
                            print(table_name)
                            em_list = [em_list[0]]
                        
                        if prop_name=='RefractiveIndex' and em_list=='':
                            print(d['anno_type'])
                        d['props'][ind][prop_row][indd] = em_list
                        
    
    pickle.dump(d, open(os.path.join(paths[i], 'unit.pkl'), 'wb'))
    return

In [8]:
for i in range(tot):
    #print(i)
    
    set_units(i)

/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:50: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
/home/scai/phd/aiz217586/.conda/envs/discomat/lib/python3.7/site-packages/ipykernel_launcher.py:152: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


In [9]:
for elem in unit_dict:
    print(f'{elem} = {unit_dict[elem]}')
    print()

Density = {'Mg/m3', 'g/cm3', 'g cm-1', 'gm/cm3', 'kg/m3', 'mgm-3', 'g/mL', 'r', 'g/cm 3', 'gm cc-1', 'O-O', 'kgm-3', 'g/cm-3', 'gw/cm3', 'kg m-3', 'd gcm-3', 'atoms/A3', 'g/cm -1', 'g/ cm3', 'g cm-3', 'gcm -3', 'gcm -1', 'g/cm', 'A', 'gm/cc', 'g/cc', 'gcm-3', 'gcm-1', 'kgm -3', 'g/l'}

GlassTransitionTg = {'ms', 'K', 'mp', 'Ref', 'mode', 's', 'T c-T g', 'degC/min', 'degC', 'degS', 'DTA', 'kJ/mol', 'K min-1'}

RefractiveIndex = {None}

AbbeValue = {None}

YoungsModulus = {'GPa', 'E'}

ShearModulus = {'GPa'}

VickersHardness = {'', 'GPa', 'N', 'Kg/mm2', 'H v', 'Gpa', 'KHN', 'HV', 'H V', 'Kgf/mm 2', 'kgfmm-2', 'kg/mm2', 'MPa', 'kgmm -2', 'Hv'}

PoissonRatio = {None}

FractureToughness = {'', 'MPam1/2', 'MPa m 1/2', 'MPa m1/2', 'IF', 'MPa m', 'Mpamm1/2', 'SENB'}

CrystallizationTemp = {'K', 'g/mol', 'AgSbS2', 's', 'degC/min', 'degC', 'T p'}

MeltingTemp = {'K', 'AgSbS2', 'h', 'T c-T g', 'degC', 'g'}

ElectricConduct = {'', 'S/m', 'O-1 cm- 1', 'O-1 m-1 K', 'O-1 cm-1', 'Sm-1', 'O-1 cm-1 K', 

In [10]:
# Density = {'', 'r', 'gcm-1', 'g/l', 'g cm-1', 'g/cm -1', 'mgm-3', 'g/ cm3', 'g/cc', 'O-O', 'g/cm3', 'gm cc-1', 'kg/m3', 'kg m-3', 'gcm-3', 'g/cm 3', 'gm/cc', 'gcm -1', 'kgm-3', 'A', 'g cm-3', 'gm/cm3', 'gcm -3'}

# GlassTransitionTg = {'', 's', 'C', 'mode', 'ms', 'DTA', 'T/K', 'Ref', 'kJ/mol', 'K min-1', 'K-1', 'degS', 'mp', 'degC', 'degC/min', 'K'}

# RefractiveIndex = {'', 'nm', 'ms', 'CWT', 'l p', 'n', 'l -', 'EM', 'l zl'}

# AbbeValue = {'', 'nd'}

# YoungsModulus = {'', 'GPa'}

# ShearModulus = {'GPa'}

# VickersHardness = {'', 'Hv', 'Kgf/mm 2', 'HV', 'Kg/mm2', 'GPa', 'kgfmm-2', 'H V', 'kg/mm2', 'kgmm -2'}

# PoissonRatio = {'', 'GPa', 'N', 'mVs-1'}

# FractureToughness = {'', 'MPa m', 'MPa m1/2', 'degC', 'MPam1/2'}

# CrystallizationTemp = {'', 'C', 'T/K', 'degC', 'degC/min', 'K'}

# MeltingTemp = {'', 'h', 'AgSbS2', 'degC', 'K'}

# ElectricConduct = {'', 'J/m2', 'l p', 'MO', 'nm', 'o', 'deg', 'Ocm', 'cm2', 'S/m', 'gcm-3', 'A-1', 'cm 2', 'MPa', 'O cm', 'O-1 cm- 1', 'A', 'O-1 cm-1', 'Scm-1', 'NdO'}

# DielectricConst = {'', 'E-0', 'o', 'GHz'}

# TSofP = {'degC', 'K'}

# TAnnealingP = {'degC', 'K'}

# ExpansionCoeff = {'', 'degC-1'}

# LiquidusTemperature = {'', 'degC', 'C', 'K'}

# SofP = {'degC', 'K'}

# BulkModulus = {'', 'GPa'}

# ActivationEnergy = {'', 'mV', 'kJmol-1', 'eV/at', 'eV', 'ac', 'kJ/mol', 'kcal/mol', 'dc'}

In [11]:
et1 = {'', 'ac', 'kJ/mol', 'dc', 'mV', 'eV/at', 'kJmol-1', 'kcal/mol', 'eV'}
et2 = {'', 'mV', 'kJmol-1', 'eV/at', 'eV', 'ac', 'kJ/mol', 'kcal/mol', 'dc'}
print(et1-et2)

set()


In [12]:
uni = 'kb'
if uni=='' or uni not in ['GPa', 'Mpa']:
    print('alala')

alala
